# Task 3: Recommendation System Analysis

## Objective

The objective of this task is to develop a content-based Netflix recommendation system using TF-IDF and cosine similarity. The system will recommend titles based on similarities in their content characteristics, such as content type, categories, directors, countries, ratings, and audience classification.

The task follows five main steps:

1. Extract relevant content features.
2. Perform text preprocessing.
3. Calculate content similarity.
4. Generate recommendations for selected titles.
5. Evaluate recommendation quality.

In [12]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# Load the cleaned dataset
df_eda = pd.read_csv("../data/netflix_clean.csv")

print("Dataset shape:", df_eda.shape)

Dataset shape: (8786, 26)


## Preparing list-based columns

The cleaned dataset contains structured list-based features for countries, genres, and directors.

Because CSV files store these lists as text, I convert them back into Python lists before using them in the recommendation analysis.

This allows individual genres, countries, and directors to be combined and analysed as separate content features.

In [13]:
# Convert list-like text back into Python lists

list_columns = [
    "country_list",
    "genre_list",
    "director_list"
]

for col in list_columns:
    df_eda[col] = df_eda[col].apply(ast.literal_eval)

print("List column types:")

for col in list_columns:
    print(f"{col}: {type(df_eda[col].iloc[0])}")

List column types:
country_list: <class 'list'>
genre_list: <class 'list'>
director_list: <class 'list'>


## Step 1: Extract relevant content features

I begin by selecting features that describe the characteristics of each Netflix title.

Because the dataset does not contain user viewing history or user-item interaction data, I use a content-based recommendation approach. Recommendations will therefore be based on similarities between the attributes of the titles themselves.

The selected content features are `type`, `genre_list`, `director_list`, `country_list`, `rating`, and `audience`.

The `show_id` and `title` columns are retained as identifiers. They are not used when calculating content similarity.

Other columns are excluded because they are either redundant with the selected features, describe catalogue metadata, or contain derived values that do not directly represent content characteristics.

In [14]:
# Select features for the recommendation system

recommendation_columns = [
    "show_id",
    "title",
    "type",
    "genre_list",
    "director_list",
    "country_list",
    "rating",
    "audience"
]

df_recommend = df_eda[recommendation_columns].copy()

print("Recommendation dataset shape:", df_recommend.shape)

print("\nSelected columns:")
display(df_recommend.head(3))

Recommendation dataset shape: (8786, 8)

Selected columns:


,show_id,title,type,genre_list,director_list,country_list,rating,audience
0,s1,Dick Johnson Is Dead,Movie,[Documentaries],[Kirsten Johnson],[United States],PG-13,Teens
1,s3,Ganglands,TV Show,"[Crime TV Shows, International TV Shows, TV Ac...",[Julien Leclercq],[France],TV-MA,Adults
2,s6,Midnight Mass,TV Show,"[TV Dramas, TV Horror, TV Mysteries]",[Mike Flanagan],[United States],TV-MA,Adults


## Step 2: Perform text preprocessing

The selected recommendation features contain both individual categorical values and list-based values.

I first convert the list-based fields for genres, directors, and countries into text. I then combine the selected content features into a single text representation for each title.

The text is converted to lowercase and unnecessary punctuation is removed so that equivalent words are treated consistently.

The resulting `content_text` field will be used for TF-IDF feature extraction in the next step.

In [15]:
# Convert list-based features into text

for col in ["genre_list", "director_list", "country_list"]:
    df_recommend[col] = df_recommend[col].apply(
        lambda values: " ".join(values)
    )


# Combine the selected content features into one text field

df_recommend["content_text"] = (
    df_recommend["type"].fillna("") + " " +
    df_recommend["genre_list"].fillna("") + " " +
    df_recommend["director_list"].fillna("") + " " +
    df_recommend["country_list"].fillna("") + " " +
    df_recommend["rating"].fillna("") + " " +
    df_recommend["audience"].fillna("")
)


# Convert to lowercase and remove unnecessary punctuation

df_recommend["content_text"] = (
    df_recommend["content_text"]
    .str.lower()
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


print("Preprocessed content examples:")

display(
    df_recommend[
        [
            "title",
            "content_text"
        ]
    ].head(10)
)

Preprocessed content examples:


,title,content_text
0,Dick Johnson Is Dead,movie documentaries kirsten johnson united sta...
1,Ganglands,tv show crime tv shows international tv shows ...
2,Midnight Mass,tv show tv dramas tv horror tv mysteries mike ...
3,Confessions of an Invisible Girl,movie children family movies comedies bruno ga...
4,Sankofa,movie dramas independent movies international ...
5,The Great British Baking Show,tv show british tv shows reality tv andy devon...
6,The Starling,movie comedies dramas theodore melfi united st...
7,Motu Patlu in the Game of Zones,movie children family movies comedies music mu...
8,Je Suis Karl,movie dramas international movies christian sc...
9,Motu Patlu in Wonderland,movie children family movies music musicals su...


## Step 3: Calculate content similarity

I use TF-IDF to convert the preprocessed content text into numerical feature vectors.

TF-IDF gives greater importance to terms that help distinguish titles from one another while reducing the influence of terms that appear frequently across the dataset.

I then calculate cosine similarity between the TF-IDF vectors. This produces a similarity score between titles based on their content features.

Higher cosine similarity indicates that two titles have more similar content representations.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# Create the TF-IDF vectorizer

tfidf = TfidfVectorizer(
    stop_words="english"
)


# Convert content text into TF-IDF features

tfidf_matrix = tfidf.fit_transform(
    df_recommend["content_text"]
)


print("TF-IDF matrix shape:", tfidf_matrix.shape)


# Calculate cosine similarity between all titles

cosine_sim = cosine_similarity(tfidf_matrix)


print("Cosine similarity matrix shape:", cosine_sim.shape)

TF-IDF matrix shape: (8786, 6552)
Cosine similarity matrix shape: (8786, 8786)


## Step 4: Generate recommendations for selected titles

I now use the cosine similarity matrix to generate recommendations for a selected Netflix title.

For a given title, I identify its position in the dataset, retrieve its similarity scores against all other titles, and rank the titles from most similar to least similar.

The selected title itself is excluded from the recommendations because every title has a perfect similarity score of 1.0 with itself.

The function returns the top recommended titles together with their similarity scores.

In [17]:
# Create a mapping from title to its position in the similarity matrix

title_indices = pd.Series(
    df_recommend.index,
    index=df_recommend["title"]
).drop_duplicates()


def recommend_titles(title, n=10):
    """
    Return the top n titles most similar to the selected title.
    """

    if title not in title_indices:
        return f"Title not found: {title}"

    # Get the row index of the selected title
    idx = title_indices[title]

    # Get similarity scores for the selected title
    similarity_scores = list(
        enumerate(cosine_sim[idx])
    )

    # Sort from highest to lowest similarity
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the selected title itself
    similarity_scores = [
        item for item in similarity_scores
        if item[0] != idx
    ]

    # Select the top n recommendations
    top_scores = similarity_scores[:n]

    # Build the recommendation table
    recommendations = pd.DataFrame({
        "title": [
            df_recommend.iloc[i]["title"]
            for i, score in top_scores
        ],
        "similarity_score": [
            round(score, 4)
            for i, score in top_scores
        ]
    })

    return recommendations

In [18]:
# Generate recommendations for Midnight Mass

recommend_titles(
    "Midnight Mass",
    n=10
)

,title,similarity_score
0,Gerald's Game,0.8056
1,Hush,0.7570
2,Brand New Cherry Flavor,0.6911
3,The Haunting of Bly Manor,0.6911
4,Ratched,0.6911
5,The Haunting of Hill House,0.6911
6,Before I Wake,0.6884
7,American Horror Story,0.6355
8,Hannibal,0.6355
9,The Originals,0.6315


## Step 5: Evaluate recommendation quality

The dataset does not contain user viewing history, ratings, or user-item interaction data, so there is no ground-truth set of recommendations against which the model can be directly compared.

I therefore evaluate the recommendations using simple content-based checks. For selected titles, I inspect the recommended titles and compare their genre overlap and content type with the original title.

This provides a basic sanity check of whether the recommendations are consistent with the content features used by the model.

The evaluation is treated as a qualitative and content-overlap assessment rather than a formal recommender-system accuracy metric.

In [19]:
def evaluate_recommendations(title, n=10):
    original_idx = title_indices[title]

    original_genres = set(
        df_eda.loc[original_idx, "genre_list"]
    )

    original_type = df_eda.loc[original_idx, "type"]

    recommendations = recommend_titles(title, n=n).copy()

    genre_overlaps = []
    type_matches = []

    for recommended_title in recommendations["title"]:
        rec_idx = title_indices[recommended_title]

        recommended_genres = set(
            df_eda.loc[rec_idx, "genre_list"]
        )

        intersection = original_genres.intersection(
            recommended_genres
        )

        union = original_genres.union(
            recommended_genres
        )

        genre_jaccard = (
            len(intersection) / len(union)
            if union
            else 0
        )

        genre_overlaps.append(round(genre_jaccard, 3))

        type_matches.append(
            df_eda.loc[rec_idx, "type"] == original_type
        )

    recommendations["genre_overlap"] = genre_overlaps
    recommendations["same_type"] = type_matches

    return recommendations


evaluation = evaluate_recommendations(
    "Midnight Mass",
    n=10
)

display(evaluation)

,title,similarity_score,genre_overlap,same_type
0,Gerald's Game,0.8056,0.0,False
1,Hush,0.7570,0.0,False
2,Brand New Cherry Flavor,0.6911,1.0,True
3,The Haunting of Bly Manor,0.6911,1.0,True
4,Ratched,0.6911,1.0,True
5,The Haunting of Hill House,0.6911,1.0,True
6,Before I Wake,0.6884,0.0,False
7,American Horror Story,0.6355,0.5,True
8,Hannibal,0.6355,0.5,True
9,The Originals,0.6315,1.0,True


### Evaluation findings

The recommendation results show that the model can identify titles with similar content characteristics.

For the `Midnight Mass` example, several recommendations have strong genre overlap, including `Brand New Cherry Flavor`, `The Haunting of Bly Manor`, `Ratched`, `The Haunting of Hill House`, and `The Originals`.

However, some highly ranked recommendations, such as `Gerald's Game` and `Hush`, have no direct genre overlap with `Midnight Mass`. This indicates that the similarity calculation is influenced by other features included in the combined text representation, such as director, country, rating, audience, and content type.

Overall, the model provides reasonable content-based recommendations, but the evaluation is limited because the dataset does not contain user viewing history or user preference data. Therefore, the results should be interpreted as a similarity-based recommendation system rather than a measure of user preference or recommendation accuracy.

In [20]:
print("Recommendation evaluation for: Midnight Mass")
print()

print(
    "Average similarity score:",
    round(evaluation["similarity_score"].mean(), 4)
)

print(
    "Recommendations with genre overlap:",
    (evaluation["genre_overlap"] > 0).sum(),
    "out of",
    len(evaluation)
)

print(
    "Recommendations with same content type:",
    evaluation["same_type"].sum(),
    "out of",
    len(evaluation)
)

print(
    "Recommendations with complete genre overlap:",
    (evaluation["genre_overlap"] == 1).sum(),
    "out of",
    len(evaluation)
)

Recommendation evaluation for: Midnight Mass

Average similarity score: 0.6918
Recommendations with genre overlap: 7 out of 10
Recommendations with same content type: 7 out of 10
Recommendations with complete genre overlap: 5 out of 10


## Final Summary

The recommendation system successfully generated content-based recommendations from the cleaned Netflix dataset.

The system used content type, genres, directors, countries, ratings, and audience classification as recommendation features. These features were converted into TF-IDF vectors, and cosine similarity was used to identify titles with similar content characteristics.

For the `Midnight Mass` example, the top 10 recommendations had an average similarity score of 0.6918. Seven of the ten recommendations shared at least one genre with the selected title, seven were the same content type, and five had complete genre overlap.

These results indicate that the system can identify reasonably similar Netflix titles based on their available content characteristics.

However, the evaluation has an important limitation. The dataset does not contain user viewing history, ratings, or user-item interaction data. Therefore, the system cannot be evaluated using standard user-preference metrics such as precision@k or recall@k. The evaluation performed here is instead a content-based sanity check using similarity, genre overlap, and content type.

Overall, the project demonstrates the basic workflow of a content-based recommendation system: feature extraction, text preprocessing, TF-IDF feature extraction, cosine similarity, recommendation generation, and evaluation.